In [ ]:
import json
import hashlib
import pandas as pd
from datetime import datetime, timezone, date

In [ ]:
SILVER_RUN_TS = datetime.now(timezone.utc).replace(tzinfo=None)

def safe_get(record, possible_keys, default=None):
    for key in possible_keys:
        if key in record and record[key] not in [None, ""]:
            return record[key]
    return default

def parse_float(value):
    try:
        return float(value)
    except Exception:
        return None

def parse_date(value):
    if value in [None, ""]:
        return None

    if isinstance(value, date) and not isinstance(value, datetime):
        return value

    if isinstance(value, datetime):
        return value.date()

    try:
        return datetime.fromisoformat(str(value).replace("Z", "+00:00")).date()
    except Exception:
        return None

def parse_timestamp(value):
    if value in [None, ""]:
        return None

    if isinstance(value, datetime):
        return value.replace(tzinfo=None)

    try:
        return datetime.fromisoformat(
            str(value).replace("Z", "+00:00")
        ).replace(tzinfo=None)
    except Exception:
        return None

def record_hash(record):
    stable_json = json.dumps(record, sort_keys=True, default=str)
    return hashlib.sha256(stable_json.encode("utf-8")).hexdigest()

In [ ]:
con = sm.get_database_connection("public_health_arrowlake")

print("Connected to ArrowLake DB")

In [ ]:
bronze_df = con.execute("""
SELECT
    run_id AS bronze_run_id,
    dataset_name AS dataset_name,
    dataset_id AS dataset_id,
    source_system AS source_system,
    raw_file_path AS raw_file_path,
    raw_record_json,
    ingestion_ts_utc AS ingestion_ts_utc,
    ingestion_date AS ingestion_date
FROM public_health.bronze_public_health_events
WHERE run_id NOT IN (
    SELECT DISTINCT bronze_run_id
    FROM public_health.silver_public_health_events
)
""").df()

print("Bronze rows pending for Silver:", len(bronze_df))

In [ ]:
silver_rows = []

for _, row in bronze_df.iterrows():
    record = json.loads(row["raw_record_json"])

    silver_rows.append({
        "dataset_name": row["dataset_name"],
        "dataset_id": row["dataset_id"],
        "source_system": row["source_system"],
        "source_record_hash": record_hash(record),

        # Generic public health mappings
        "jurisdiction": safe_get(record, [
            "state", "jurisdiction", "location", "locationdesc", "res_state"
        ]),

        "condition_name": safe_get(record, [
            "condition", "condition_name", "indicator", "topic", "measure"
        ]),

        "report_period": safe_get(record, [
            "week", "mmwr_week", "year", "time_period", "period"
        ]),

        "report_date": parse_date(safe_get(record, [
            "report_date", "date", "week_end", "end_date", "created_at"
        ])),

        "value": parse_float(safe_get(record, [
            "value", "data_value", "count", "cases", "rate"
        ])),

        "unit": safe_get(record, [
            "unit", "data_value_unit", "measure_unit"
        ]),

        "source_updated_at": parse_timestamp(safe_get(record, [
            ":updated_at", "updated_at", "last_updated", "created_at"
        ])),

        "raw_file_path": row["raw_file_path"],
        "bronze_run_id": row["bronze_run_id"],
        "ingestion_ts_utc": parse_timestamp(row["ingestion_ts_utc"]),
        "ingestion_date": parse_date(row["ingestion_date"]),
        "silver_created_at_utc": parse_timestamp(SILVER_RUN_TS),
    })

silver_schema = {
    "dataset_name": pl.Utf8,
    "dataset_id": pl.Utf8,
    "source_system": pl.Utf8,
    "source_record_hash": pl.Utf8,
    "jurisdiction": pl.Utf8,
    "condition_name": pl.Utf8,
    "report_period": pl.Utf8,
    "report_date": pl.Date,
    "value": pl.Float64,
    "unit": pl.Utf8,
    "source_updated_at": pl.Datetime,
    "raw_file_path": pl.Utf8,
    "bronze_run_id": pl.Utf8,
    "ingestion_ts_utc": pl.Datetime,
    "ingestion_date": pl.Date,
    "silver_created_at_utc": pl.Datetime,
}

# silver_df = pl.DataFrame(
#     silver_rows,
#     schema=silver_schema,
#     strict=False
# )

silver_df = pl.DataFrame(
    silver_rows,
    schema=silver_schema,
    strict=False,
    infer_schema_length=None
)

print("Silver candidate rows:", len(silver_df))
silver_df.head()

In [ ]:
if not silver_df.is_empty():
    con.register("silver_incoming", silver_df)

    con.execute("""
    INSERT INTO public_health.silver_public_health_events
    SELECT *
    FROM silver_incoming s
    WHERE NOT EXISTS (
        SELECT 1
        FROM public_health.silver_public_health_events t
        WHERE t.source_record_hash = s.source_record_hash
          AND t.dataset_id = s.dataset_id
    )
    """)

print("Silver load completed.")

In [ ]:
con.execute("""
SELECT
    dataset_name,
    dataset_id,
    COUNT(*) AS silver_rows,
    COUNT(DISTINCT source_record_hash) AS distinct_records,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date,
    MAX(silver_created_at_utc) AS last_silver_load
FROM public_health.silver_public_health_events
GROUP BY dataset_name, dataset_id
ORDER BY last_silver_load DESC
""").df()

In [ ]:
con.execute("""
SELECT *
FROM public_health.silver_public_health_events
ORDER BY silver_created_at_utc DESC
LIMIT 20
""").df()